# Where does the timing predictor fail?

Steps 1–2 only: review saved held-example errors and candidate geometric explanations. **No models are fitted here.** The reference is the length-context model with SD, logged remaining ratios and Elastic Net; the comparison is the matching original ratio model. Both were evaluated on identical records.

Residual $e=\widehat{\Delta t}-\Delta t$: **positive** means the predicted child peak is too late relative to the parent (or its advance is underestimated); **negative** means too early. All errors below are minutes. Default `all` includes 54; switch to `without54` for the separately refitted models.

Classification uses current state aliases and saved parent unit hierarchy. Full-compound aliases take precedence, followed by primary island edits, compound members, standalone nested units, then simple units. An island alias of an identical unit edit is not a second observation. Nesting is also retained as an independent flag because it overlaps compound membership. There are no standalone nested cases under this exclusive classification; nested member edits are still represented.

## Conditioning check: correction to the parent-length interpretation

Absolute parent-domain mean length equals relative domain size multiplied by parent-network mean length. The latter is constant within each network. Therefore the two predictors have identical within-network ranks, and **identical residual correlations in all 15 networks** (median 0.5016). The same holds in all 14 separately refitted without-54 cases. Maximum reconstruction error is below 1e-12 m.

Conditioning on relative size within a network leaves no independent absolute-length variation. A partial correlation would be undefined, rather than a meaningful zero. Adding other controls cannot restore variation already removed by this exact relation.

The previous correlation did **not** establish missing absolute-length information. It shows a residual pattern along a predictor already supplied to the model. Nonlinear effects, regularization, interactions and differing relationships across networks remain possible explanations. This does not establish which explanation is correct.

Adding absolute length would introduce between-network scale through the identity. Its usefulness across unseen networks remains untested, and is distinct from the within-network association reported here. No new predictor fits were run. The reproducible audit is `check_length_conditioning.py`, with per-network results in `parent_length_conditioning_check.csv`.


## Current findings

05 has MAE 14.01 min and bias +13.22 min: peak advances are underpredicted. It contributes 27.1% of equal-example squared error. In contrast, 48 and 54 have small observed shifts but predictions advance the peaks too far.

Full-compound collapses have MAE 19.36 min against 43.35 min for zero prediction. Simple units have MAE 8.50 min against 6.93 min for zero prediction, with bias −5.06 min. Simple-unit bias deserves attention alongside the largest compound errors.

Parent domain mean route length has positive residual association in 14/15 networks (median within-network Spearman rho 0.50). After aggregating repeated edits to 435 parent outer-domain graphs, all 12 networks with sufficient variation have positive correlations (median 0.78). This does not establish causal importance or guarantee predictive improvement.

Correction after conditioning check: the parent-length residual association duplicates the association with the existing relative-size input within each network. It is not independent evidence for adding absolute length. Expected bifurcations, cycle rank and total split impact are secondary candidates; they overlap with size and type. No additional model fitting has been performed.

In [9]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown,display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src/rapid_tools').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_residual_review_20261018'
pio.renderers.default='plotly_mimetype'
SCENARIO='all'
d=pd.read_parquet(OUT/'diagnostic_rows.parquet');d=d[d.scenario.eq(SCENARIO)].copy()
def read(name):
    z=pd.read_csv(OUT/name,dtype={'example':str})
    return z[z.scenario.eq(SCENARIO)].copy() if 'scenario' in z else z
def table(z):
    lines=['| '+' | '.join(map(str,z.columns))+' |','| '+' | '.join(['---']*len(z.columns))+' |']
    for row in z.itertuples(index=False,name=None):lines.append('| '+' | '.join(str(x).replace('|',' / ') for x in row)+' |')
    display(Markdown('\n'.join(lines)))
net=read('network_errors.csv').sort_values('mae_min',ascending=False)
table(net[['example','n','bias_min','mae_min','rmse_min','mean_abs_shift_min','mae_vs_zero','equal_example_squared_error_share_pct']].round(3))

| example | n | bias_min | mae_min | rmse_min | mean_abs_shift_min | mae_vs_zero | equal_example_squared_error_share_pct |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 05 | 51 | 13.216 | 14.009 | 23.004 | 21.784 | 0.643 | 27.068 |
| 13 | 66 | 6.4 | 8.054 | 12.907 | 14.182 | 0.568 | 8.521 |
| 54 | 81 | -7.157 | 7.784 | 9.568 | 3.037 | 2.563 | 4.682 |
| 48 | 72 | -7.308 | 7.308 | 9.715 | 1.708 | 4.278 | 4.828 |
| 46 | 223 | 2.408 | 7.004 | 13.834 | 10.215 | 0.686 | 9.789 |
| 21 | 30 | -6.344 | 6.45 | 7.283 | 5.3 | 1.217 | 2.713 |
| 41 | 181 | 4.685 | 5.862 | 13.71 | 10.05 | 0.583 | 9.615 |
| 45 | 93 | 3.539 | 5.166 | 8.955 | 10.849 | 0.476 | 4.102 |
| 03 | 40 | -4.565 | 5.018 | 6.798 | 5.875 | 0.854 | 2.364 |
| 02 | 35 | -2.96 | 4.683 | 6.591 | 6.371 | 0.735 | 2.222 |
| 56 | 21 | -0.995 | 4.523 | 6.082 | 14.381 | 0.315 | 1.892 |
| 55 | 212 | 3.349 | 4.511 | 9.67 | 8.608 | 0.524 | 4.783 |
| 37 | 395 | -0.469 | 4.143 | 14.222 | 6.225 | 0.665 | 10.346 |
| 15 | 262 | -3.084 | 3.995 | 10.558 | 3.225 | 1.239 | 5.702 |
| 22 | 408 | -0.981 | 2.152 | 5.181 | 2.647 | 0.813 | 1.373 |

## 1. Network and edit-type error overview

Compare MAE with the mean absolute observed shift (the zero-prediction baseline). `mae_vs_zero > 1` means the model is worse than predicting no timing change on that subset. This denominator is averaged over the group, not a percentage error for near-zero individual shifts. Squared-error shares use equal total weight per network.

Type summaries equal-weight the networks represented in each category. Different categories may contain different network mixtures, so consult the network-by-type tables too. Median and 90th-percentile errors are unweighted descriptive quantiles.

In [10]:
plot=net.melt(id_vars=['example'],value_vars=['mae_min','mean_abs_shift_min','bias_min'],var_name='quantity',value_name='minutes')
fig=px.bar(plot,x='example',y='minutes',color='quantity',barmode='group',template='plotly_white',category_orders={'example':net.example.tolist()},title='Network errors versus observed shift magnitude')
fig.show()
table(read('edit_type_errors.csv')[['edit_type','n','examples','bias_min','mae_min','rmse_min','mean_abs_shift_min','mae_vs_zero']].round(3))
table(read('nested_strata_errors.csv')[['edit_type','nested','n','examples','bias_min','mae_min']].round(3))

| edit_type | n | examples | bias_min | mae_min | rmse_min | mean_abs_shift_min | mae_vs_zero |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Compound member | 1467 | 15 | 0.689 | 3.817 | 7.907 | 5.315 | 0.718 |
| Full compound | 177 | 15 | 3.437 | 19.362 | 29.655 | 43.348 | 0.447 |
| Island | 268 | 12 | -1.072 | 5.376 | 11.457 | 7.917 | 0.679 |
| Simple | 258 | 15 | -5.063 | 8.499 | 10.097 | 6.925 | 1.227 |

| edit_type | nested | n | examples | bias_min | mae_min |
| --- | --- | --- | --- | --- | --- |
| Compound member | False | 706 | 15 | 0.418 | 3.667 |
| Compound member | True | 761 | 15 | 0.529 | 3.276 |
| Full compound | False | 96 | 13 | 1.014 | 12.205 |
| Full compound | True | 81 | 15 | 0.638 | 22.063 |
| Island | False | 8 | 3 | -1.558 | 3.098 |
| Island | True | 260 | 12 | -1.356 | 5.69 |
| Simple | False | 258 | 15 | -5.063 | 8.499 |

In [11]:
centroid_plot

,example,state,cohort,observed_hours,predicted_hours,edit_type,centroid_shift_hours,peak_shift_hours,observed_min,predicted_min,centroid_error_min,modelled_peak_shift_min,peak_shift_proxy_min,peak_proxy_error_min,observed_signed_log,predicted_signed_log
0,02,CB001_bubble_5,original,-0.441575,-0.186433,Full compound,-0.441575,-0.850000,-26.494524,-11.185953,15.308571,-51.0,-11.185953,39.814047,-1.439246,-1.085859
1,02,CB002_bubble_2,original,-0.239604,-0.122718,Full compound,-0.239604,-0.500000,-14.376245,-7.363060,7.013186,-30.0,-7.363060,22.636940,-1.186850,-0.922365
2,02,CB003_bubble_4,original,-0.143013,-0.151630,Full compound,-0.143013,-0.033333,-8.580758,-9.097772,-0.517014,-2.0,-9.097772,-7.097772,-0.981400,-1.004226
3,02,CB004_bubble_1,original,-0.071236,-0.037615,Full compound,-0.071236,-0.133333,-4.274174,-2.256923,2.017251,-8.0,-2.256923,5.743077,-0.722154,-0.512808
4,02,CB005_bubble_6,original,-0.113398,-0.049131,Full compound,-0.113398,-0.233333,-6.803878,-2.947858,3.856021,-14.0,-2.947858,11.052142,-0.892310,-0.596361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2180,56,S015_unit_4,original,-0.545294,-0.554167,Compound member,-0.545294,-0.600000,-32.717658,-33.250044,-0.532386,-36.0,-33.250044,2.749956,-1.527857,-1.534661
2181,56,all_closed_units_collapsed,segmented,-2.537611,-1.812498,Segmented full network,-2.537611,-2.800000,-152.256641,-108.749904,43.506736,-168.0,-108.749904,59.250096,-2.185419,-2.040404
2182,56,island_56_005,original,-0.307500,-0.340227,Island,-0.307500,-0.333333,-18.450003,-20.413603,-1.963600,-20.0,-20.413603,-0.413603,-1.288920,-1.330690
2183,56,island_56_011,original,-0.448790,-0.389312,Island,-0.448790,-0.466667,-26.927403,-23.358697,3.568706,-28.0,-23.358697,4.641303,-1.446031,-1.386654


In [12]:
import numpy as np
import pandas as pd
import plotly.express as px

# Current depth-informed model: 10 predictors.
# Each held-out network uses training-only tuning.
# Evaluation predictions come from nested validation, not the all-data refit.
RESULTS = ROOT / "RAPID/outputs/uncapped_event_window_20261020"
CENTROID_MODEL = "nested_tuned"

# Plot selections:
COHORT = "all"       # "all", "original", or "segmented"
EXCLUDE_54 = False   # Scoring/plotting exclusion only; no retraining

centroid_predictions = pd.read_parquet(
    RESULTS / "ten_input_tuning_20261023/predictions.parquet"
)
current_metadata = pd.read_parquet(
    RESULTS / "alternative_validation_20261023/input.parquet"
)

centroid_plot = centroid_predictions.loc[
    centroid_predictions.variant.eq(CENTROID_MODEL),
    ["example", "state", "cohort", "centroid_shift_hours", "predicted_hours"],
].rename(columns={"centroid_shift_hours": "observed_hours"}).copy()

centroid_plot = centroid_plot.merge(
    current_metadata[
        [
        "example", "state", "cohort", "edit_type",
        "centroid_shift_hours", "peak_shift_hours",
        ]
    ],
    on=["example", "state", "cohort"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)
assert centroid_plot["_merge"].eq("both").all(), (
    "Prediction and diagnostic rows differ:\n"
    + centroid_plot.loc[
        centroid_plot["_merge"].ne("both"),
        ["example", "state", "cohort", "_merge"],
    ].to_string(index=False)
)
centroid_plot = centroid_plot.drop(columns="_merge")

np.testing.assert_allclose(
    centroid_plot.observed_hours,
    centroid_plot.centroid_shift_hours,
)

assert COHORT in {"all", "original", "segmented"}
if COHORT != "all":
    centroid_plot = centroid_plot.loc[
        centroid_plot.cohort.eq(COHORT)
    ].copy()
if EXCLUDE_54:
    centroid_plot = centroid_plot.loc[
        centroid_plot.example.ne("54")
    ].copy()

assert len(centroid_plot) > 0, "No centroid predictions for this selection"

centroid_plot["observed_min"] = 60 * centroid_plot.observed_hours
centroid_plot["predicted_min"] = 60 * centroid_plot.predicted_hours
centroid_plot["centroid_error_min"] = (
    centroid_plot.predicted_min - centroid_plot.observed_min
)
# Peak timing from RAPID: child global-maximum time minus parent.
centroid_plot["modelled_peak_shift_min"] = (
    60 * centroid_plot["peak_shift_hours"]
)

# Existing held-out centroid prediction used as a peak-shift proxy.
# This is NOT a separately trained peak-timing prediction.
centroid_plot["peak_shift_proxy_min"] = centroid_plot["predicted_min"]

centroid_plot["peak_proxy_error_min"] = (
    centroid_plot["peak_shift_proxy_min"]
    - centroid_plot["modelled_peak_shift_min"]
)

assert np.isfinite(
    centroid_plot[["observed_min", "predicted_min"]]
).all().all()

print(
    f"{len(centroid_plot):,} comparisons | "
    f"{centroid_plot.example.nunique()} networks | "
    f"{centroid_plot.cohort.eq('original').sum():,} regular | "
    f"{centroid_plot.cohort.eq('segmented').sum()} segmented full collapses"
)

# Signed log1p is continuous at zero.
# Axis ticks and hover labels retain actual minutes.
SCALE_MIN = 1.0

def signed_log(values):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.log10(1 + np.abs(values) / SCALE_MIN)

centroid_plot["observed_signed_log"] = signed_log(
    centroid_plot.observed_min
)
centroid_plot["predicted_signed_log"] = signed_log(
    centroid_plot.predicted_min
)
plot_labels = {
    "observed_min": "Observed",
    "predicted_min": "Predicted",
    "centroid_error_min": "Prediction error",
    "modelled_peak_shift_min": "Modelled peak shift",
    "peak_shift_proxy_min": "Predicted peak shift",
}

fig = px.scatter(
    centroid_plot,
    x="observed_signed_log",
    y="predicted_signed_log",
    color_discrete_sequence=["black"],
    opacity=0.5,
    labels={
    "observed_min": "Observed",
    "predicted_min": "Predicted",
    "centroid_error_min": "Prediction error",
    "modelled_peak_shift_min": "Modelled peak shift",
    "peak_shift_proxy_min": "Predicted peak shift",
    },
    hover_data={
        "observed_signed_log": False,
        "predicted_signed_log": False,
        "example": True,
        "state": True,
        "edit_type": True,
        "cohort": True,
        "observed_min": ":.3f",
        "predicted_min": ":.3f",
        "centroid_error_min": ":.3f",
        "modelled_peak_shift_min": ":.3f",
        "peak_shift_proxy_min": ":.3f",
    },
    template="plotly_white",
)

# Fit each axis independently, preserving the original plot styling.
def point_range(values):
    lo, hi = float(values.min()), float(values.max())
    padding = max((hi - lo) * 0.05, 0.01)
    return [lo - padding, hi + padding]

x_range = point_range(centroid_plot.observed_signed_log)
y_range = point_range(centroid_plot.predicted_signed_log)

positive_ticks = np.array([
    0.5, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000
])
ticks_min = np.r_[-positive_ticks[::-1], 0, positive_ticks]

axis_options = dict(
    tickmode="array",
    tickvals=signed_log(ticks_min).tolist(),
    ticktext=[f"{v:g}" for v in ticks_min],
    zeroline=True,
    zerolinecolor="lightgrey",
)

fig.update_xaxes(
    **axis_options,
    range=x_range,
    tickangle=0,
    title="RAPID centroid shift [min]",
)
fig.update_yaxes(
    **axis_options,
    range=y_range,
    title="Predicted centroid shift [min]",
)

# Equality line restricted to the visible overlap of both axes.
line_lo = max(x_range[0], y_range[0])
line_hi = min(x_range[1], y_range[1])
if line_lo < line_hi:
    fig.add_shape(
        type="line",
        x0=line_lo, y0=line_lo,
        x1=line_hi, y1=line_hi,
        line=dict(dash="dash", color="grey"),
        layer="below",
    )

fig.update_layout(
    title="Depth-informed RF — nested leave-one-network-out",
    width=750,
    height=700,
)
fig.show()

2,185 comparisons | 15 networks | 2,170 regular | 15 segmented full collapses


In [13]:
cross=read('network_edit_errors.csv')
for value,title in [('mae_min','Mean absolute error [min]'),('n','Comparison counts')]:
    matrix=cross.pivot(index='example',columns='edit_type',values=value)
    fig=px.imshow(matrix,text_auto='.1f' if value=='mae_min' else '.0f',aspect='auto',
        color_continuous_scale='YlOrRd',title=title)
    fig.update_layout(height=550);fig.show()

fig=px.scatter(d,x='observed_hours',y='predicted_hours',color='example',symbol='edit_type',
        hover_data=['state','error_min'],template='plotly_white',
        title='Held-example predictions: full range, including extreme cases')
lo=min(d.observed_hours.min(),d.predicted_hours.min());hi=max(d.observed_hours.max(),
        d.predicted_hours.max())
fig.add_shape(type='line',x0=lo,y0=lo,x1=hi,y1=hi,line=dict(dash='dash',color='grey'));fig.show()

table(read('worst_cases.csv')[['example','state','edit_type','observed_hours',
        'predicted_hours','error_min','outside_input_range']].head(15).round(3))

| example | state | edit_type | observed_hours | predicted_hours | error_min | outside_input_range |
| --- | --- | --- | --- | --- | --- | --- |
| 46 | CB001_bubble_90 | Full compound | -2.6 | -0.695 | 114.278 | False |
| 37 | island_37_209 | Island | 2.117 | 0.4 | -103.017 | True |
| 37 | island_37_161 | Island | 1.9 | 0.205 | -101.711 | True |
| 46 | S178_unit_44 | Compound member | -2.3 | -0.61 | 101.387 | False |
| 37 | island_37_191 | Island | 1.867 | 0.182 | -101.084 | True |
| 37 | island_37_195 | Island | 1.867 | 0.187 | -100.795 | True |
| 05 | CB008_bubble_33 | Full compound | -2.033 | -0.358 | 100.53 | False |
| 37 | island_37_014 | Island | 1.75 | 0.179 | -94.247 | True |
| 37 | island_37_230 | Island | 1.967 | 0.417 | -92.951 | True |
| 37 | CB001_bubble_3 | Full compound | -8.75 | -7.244 | 90.367 | True |
| 15 | CB001_bubble_4 | Full compound | -2.583 | -4.023 | -86.357 | False |
| 41 | CB005_bubble_37 | Full compound | -2.167 | -0.819 | 80.867 | False |
| 41 | CB002_bubble_26 | Full compound | -1.817 | -0.502 | 78.861 | False |
| 41 | CB003_bubble_23 | Full compound | -1.917 | -0.623 | 77.634 | False |
| 55 | CB001_bubble_38 | Full compound | -1.7 | -0.443 | 75.414 | False |

## 2. Candidate explanations

These are **associations with held-out errors**, not proof of added predictive value. No row-level significance tests are used: edits share parent hydrographs and often the same compound domain.

Signed error can reveal systematic under/overprediction; absolute error identifies difficult cases but need not indicate a missing directional effect. For each omitted measure we inspect the pooled Spearman correlation, the median correlation computed separately within networks, and sign consistency. A within-network correlation requires at least 10 finite comparisons and variation in both quantities.

The next tables control for edit type descriptively and aggregate repeated edits to one median per parent outer-domain graph. They help check whether an apparent relationship only reflects category composition or repeated compound measurements. They are not conditional importance estimates.

Parent values and differences are diagnostic probes here—not newly selected predictor representations. Whole-network parent profiles are stored separately, with only 15 independent networks; they must not be treated as thousands of observations.

In [14]:
symbols={'route_length_mean': '\\mu_L', 'route_length_sd': '\\sigma_L', 'width_mean': '\\bar W', 'total_length': 'L_{\\mathrm{tot}}', 'area_elongation': '\\mu_L^2/A_{LW}', 'route_length_min': 'L_{\\min}', 'route_length_max': 'L_{\\max}', 'route_length_skew': '\\gamma_L', 'branch_length_contrast_mean': '\\bar c_L', 'branch_length_contrast_sd': '\\sigma_{c_L}', 'dominance_sd': '\\sigma_d', 'split_entropy_sd': '\\sigma_h', 'effective_branches_sd': '\\sigma_{n_{\\mathrm{eff}}}', 'expected_splits': '\\mathbb E[N_b]', 'cycle_rank': 'E-V+1', 'bifurcations': 'N_b', 'bifurcation_density': '\\rho_b', 'split_impact_sum': 'I_\\Sigma', 'split_impact_sd': '\\sigma_I', 'split_exposure_sd': '\\sigma_{r_b}', 'width_harmonic': 'W_H'}
def symbol(feature):
    _,rep,metric=feature.split('_',2)
    return '$'+(chr(92)+'Delta('+symbols[metric]+')^D' if rep=='delta' else '('+symbols[metric]+')_p^D')+'$'
a=read('residual_associations.csv')
z=a[a.error_type.eq('error_min')&a.examples_with_variation.ge(8)].sort_values('median_within_rho',key=abs,ascending=False).head(12).copy()
z['symbol']=z.feature.map(symbol)
table(z[['symbol','coverage','pooled_rho','examples_with_variation','median_within_rho','positive_examples','negative_examples']].round(3))
controls=read('within_type_associations.csv')
controls=controls[controls.feature.isin(z.feature)&controls.examples_with_variation.ge(5)].copy();controls['symbol']=controls.feature.map(symbol)
table(controls[['edit_type','symbol','examples_with_variation','median_within_rho','positive_examples','negative_examples']].round(3))
agg=read('domain_aggregated_associations.csv');agg=agg[agg.feature.isin(z.feature)].copy();agg['symbol']=agg.feature.map(symbol)
table(agg[['symbol','outer_domains','examples_with_variation','median_within_rho','positive_examples','negative_examples']].round(3))

| symbol | coverage | pooled_rho | examples_with_variation | median_within_rho | positive_examples | negative_examples |
| --- | --- | --- | --- | --- | --- | --- |
| $(L_{\mathrm{tot}})_p^D$ | 1.0 | 0.22 | 15 | 0.547 | 14 | 1 |
| $(\mu_L^2/A_{LW})_p^D$ | 1.0 | 0.203 | 15 | 0.512 | 15 | 0 |
| $(L_{\max})_p^D$ | 1.0 | 0.167 | 15 | 0.508 | 14 | 1 |
| $(\mu_L)_p^D$ | 1.0 | 0.178 | 15 | 0.502 | 14 | 1 |
| $(L_{\min})_p^D$ | 1.0 | 0.226 | 15 | 0.498 | 14 | 1 |
| $(\rho_b)_p^D$ | 1.0 | -0.264 | 15 | -0.397 | 3 | 12 |
| $(I_\Sigma)_p^D$ | 1.0 | 0.14 | 15 | 0.362 | 14 | 1 |
| $(\mathbb E[N_b])_p^D$ | 1.0 | 0.146 | 15 | 0.362 | 14 | 1 |
| $(N_b)_p^D$ | 1.0 | 0.14 | 15 | 0.361 | 14 | 1 |
| $(E-V+1)_p^D$ | 1.0 | 0.14 | 15 | 0.361 | 14 | 1 |
| $\Delta(\rho_b)^D$ | 1.0 | 0.155 | 15 | 0.358 | 14 | 1 |
| $(\sigma_{c_L})_p^D$ | 1.0 | 0.111 | 15 | 0.309 | 14 | 1 |

| edit_type | symbol | examples_with_variation | median_within_rho | positive_examples | negative_examples |
| --- | --- | --- | --- | --- | --- |
| Compound member | $(\mu_L)_p^D$ | 13 | 0.327 | 12 | 1 |
| Compound member | $(L_{\min})_p^D$ | 13 | 0.359 | 12 | 1 |
| Compound member | $(L_{\max})_p^D$ | 13 | 0.323 | 11 | 2 |
| Compound member | $(\sigma_{c_L})_p^D$ | 13 | 0.23 | 10 | 3 |
| Compound member | $(\mathbb E[N_b])_p^D$ | 13 | 0.23 | 11 | 2 |
| Compound member | $(E-V+1)_p^D$ | 13 | 0.24 | 10 | 3 |
| Compound member | $(N_b)_p^D$ | 13 | 0.24 | 10 | 3 |
| Compound member | $(\rho_b)_p^D$ | 13 | -0.208 | 4 | 9 |
| Compound member | $\Delta(\rho_b)^D$ | 13 | 0.188 | 11 | 2 |
| Compound member | $(I_\Sigma)_p^D$ | 13 | 0.23 | 11 | 2 |
| Compound member | $(\mu_L^2/A_{LW})_p^D$ | 13 | 0.331 | 12 | 1 |
| Compound member | $(L_{\mathrm{tot}})_p^D$ | 13 | 0.323 | 11 | 2 |
| Full compound | $(\mu_L)_p^D$ | 7 | 0.665 | 5 | 2 |
| Full compound | $(L_{\min})_p^D$ | 7 | 0.653 | 5 | 2 |
| Full compound | $(L_{\max})_p^D$ | 7 | 0.637 | 5 | 2 |
| Full compound | $(\sigma_{c_L})_p^D$ | 7 | 0.32 | 5 | 2 |
| Full compound | $(\mathbb E[N_b])_p^D$ | 7 | 0.16 | 4 | 3 |
| Full compound | $(E-V+1)_p^D$ | 7 | 0.301 | 4 | 3 |
| Full compound | $(N_b)_p^D$ | 7 | 0.301 | 4 | 3 |
| Full compound | $(\rho_b)_p^D$ | 7 | -0.534 | 1 | 6 |
| Full compound | $\Delta(\rho_b)^D$ | 7 | 0.534 | 6 | 1 |
| Full compound | $(I_\Sigma)_p^D$ | 7 | 0.424 | 4 | 3 |
| Full compound | $(\mu_L^2/A_{LW})_p^D$ | 7 | 0.521 | 5 | 2 |
| Full compound | $(L_{\mathrm{tot}})_p^D$ | 7 | 0.603 | 5 | 2 |
| Island | $(\mu_L)_p^D$ | 6 | 0.077 | 5 | 1 |
| Island | $(L_{\min})_p^D$ | 6 | 0.077 | 5 | 1 |
| Island | $(L_{\max})_p^D$ | 6 | 0.097 | 5 | 1 |
| Island | $(\sigma_{c_L})_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(\mathbb E[N_b])_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(E-V+1)_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(N_b)_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(\rho_b)_p^D$ | 6 | -0.021 | 3 | 3 |
| Island | $\Delta(\rho_b)^D$ | 6 | 0.072 | 4 | 2 |
| Island | $(I_\Sigma)_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(\mu_L^2/A_{LW})_p^D$ | 6 | 0.05 | 4 | 2 |
| Island | $(L_{\mathrm{tot}})_p^D$ | 6 | 0.05 | 5 | 1 |
| Simple | $(\mu_L)_p^D$ | 9 | 0.833 | 9 | 0 |
| Simple | $(L_{\min})_p^D$ | 9 | 0.831 | 9 | 0 |
| Simple | $(L_{\max})_p^D$ | 9 | 0.749 | 9 | 0 |
| Simple | $(\rho_b)_p^D$ | 9 | -0.808 | 0 | 9 |
| Simple | $\Delta(\rho_b)^D$ | 9 | 0.808 | 9 | 0 |
| Simple | $(I_\Sigma)_p^D$ | 9 | 0.27 | 6 | 3 |
| Simple | $(\mu_L^2/A_{LW})_p^D$ | 9 | 0.8 | 9 | 0 |
| Simple | $(L_{\mathrm{tot}})_p^D$ | 9 | 0.808 | 9 | 0 |

| symbol | outer_domains | examples_with_variation | median_within_rho | positive_examples | negative_examples |
| --- | --- | --- | --- | --- | --- |
| $(\mu_L)_p^D$ | 435 | 12 | 0.781 | 12 | 0 |
| $(L_{\min})_p^D$ | 435 | 12 | 0.782 | 12 | 0 |
| $(L_{\max})_p^D$ | 435 | 12 | 0.753 | 12 | 0 |
| $(\sigma_{c_L})_p^D$ | 435 | 12 | 0.62 | 11 | 1 |
| $(\mathbb E[N_b])_p^D$ | 435 | 12 | 0.653 | 11 | 1 |
| $(E-V+1)_p^D$ | 435 | 12 | 0.64 | 11 | 1 |
| $(N_b)_p^D$ | 435 | 12 | 0.64 | 11 | 1 |
| $(\rho_b)_p^D$ | 435 | 12 | -0.532 | 3 | 9 |
| $\Delta(\rho_b)^D$ | 435 | 12 | 0.746 | 12 | 0 |
| $(I_\Sigma)_p^D$ | 435 | 12 | 0.553 | 12 | 0 |
| $(\mu_L^2/A_{LW})_p^D$ | 435 | 12 | 0.755 | 12 | 0 |
| $(L_{\mathrm{tot}})_p^D$ | 435 | 12 | 0.767 | 12 | 0 |

In [15]:
d.columns

Index(['scenario', 'pipeline', 'variant', 'example', 'state', 'observed_hours',
       'predicted_hours', 'observed_pct', 'predicted_pct',
       'baseline_prediction',
       ...
       'error_min', 'abs_error_min', 'baseline_error_min',
       'log1p__B_ratio_route_length_mean', 'log1p__B_ratio_route_length_sd',
       'log1p__B_ratio_route_length_cv', 'log1p__B_ratio_width_mean',
       'log1p__B_ratio_width_cv', 'log1p__B_ratio_route_entropy',
       'outside_input_range'],
      dtype='object', length=681)

In [16]:
FEATURE='B_parent_total_length'  # Change to any audited candidate column.
fig=px.scatter(d,x=FEATURE,y='error_min',color='example',symbol='edit_type',hover_data=['state','observed_hours','predicted_hours'],template='plotly_white',title='Residual versus '+symbol(FEATURE))
fig.add_hline(y=0,line_dash='dot');fig.show()
# Per-network rank plots make differing geometric scales easier to compare.
z=d.copy();z['within_network_rank']=z.groupby('example')[FEATURE].rank(pct=True)
fig=px.scatter(z,x='within_network_rank',y='error_min',facet_col='example',facet_col_wrap=5,color='edit_type',hover_data=['state'],template='plotly_white',title='Within-network predictor rank versus signed error')
fig.update_layout(height=850);fig.show()
support=read('training_range_audit.csv')
table(support[support.outside_fraction.gt(.1)][['example','feature','outside_fraction','train_min','train_max']].round(4))

| example | feature | outside_fraction | train_min | train_max |
| --- | --- | --- | --- | --- |
| 37 | B_mean_length_fraction | 0.9696 | 0.0011 | 0.3722 |

## Interpretation and next decision

The evidence can justify a short list of hypotheses, not yet additions to the final predictor. A variable may correlate with errors because of omitted physics, an inadequate functional form, or its association with existing predictors. Candidate additions still require controlled refitting on identical grouped validation splits.

All saved comparison keys and target values are checked against the current geometry/response table. Classification provenance and aliases are in `classification_audit.csv`; formulas and caveats are recorded in `manifest.json`. The original model outputs remain unchanged.

## Is the error pattern model-dependent?

Compare Ridge, Elastic Net, random forest and boosting on exactly the same SD/log1p length-context inputs. All predictions are held-example estimates; each family uses its own inner-tuned settings. No new models are fitted.

The main network biases persist: all four underpredict advances in 05 and overpredict them in 48 and 54. Model choice still matters: boosting is much worse in 03, and the linear models perform better overall. Ridge and Elastic Net are especially similar.

Shared residual patterns make a purely Elastic-Net-specific explanation less plausible, but do not establish missing physics. Models share training data, inputs and the observed target; common support limitations and fitting constraints remain alternative explanations. Error correlations are descriptive, not independent statistical evidence.

In [17]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src/rapid_tools').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_residual_review_20261018'
pio.renderers.default='plotly_mimetype'
SCENARIO='all'
m=pd.read_csv(OUT/'model_network_errors.csv',dtype={'example':str});m=m[m.scenario.eq(SCENARIO)]
for value,title in [('bias_min','Mean signed error [min]'),('mae_min','Mean absolute error [min]')]:
    matrix=m.pivot(index='example',columns='model',values=value)
    fig=px.imshow(matrix,text_auto='.1f',aspect='auto',color_continuous_scale='RdBu_r' if value=='bias_min' else 'YlOrRd',color_continuous_midpoint=0 if value=='bias_min' else None,title=title+' — identical inputs')
    fig.update_layout(height=550);fig.show()
corr=pd.read_csv(OUT/'model_error_correlations.csv');display(corr[corr.scenario.eq(SCENARIO)].round(3))
agreement=pd.read_csv(OUT/'model_agreement_summary.csv');display(agreement.round(3))


,scenario,model_a,model_b,equal_example_error_correlation,median_within_example_correlation
0,all,ridge,elastic_net,0.997,0.998
1,all,ridge,forest,0.815,0.846
2,all,ridge,boosting,0.736,0.749
3,all,elastic_net,forest,0.807,0.833
4,all,elastic_net,boosting,0.728,0.722
5,all,forest,boosting,0.868,0.877


,scenario,n,equal_example_same_sign_fraction,equal_example_common_error_over_5min_fraction
0,all,2170,0.773,0.231
1,without54,2089,0.728,0.212


## Advances, unchanged peaks and delays

Saved held-out predictions from the SD/log1p length-context Elastic Net model. Groups use observed global-maximum timing: negative = advance, zero = same saved peak time, positive = delay. No refitting or exclusions. Networks represented in each group receive equal total weight; group-specific weighting differs from the overall score. These are diagnostic subsets, not separately validated deployment tasks.

| Outcome | Cases | Networks | R² | MAE [min] | Bias [min] | Zero-prediction MAE [min] |
|---|---:|---:|---:|---:|---:|---:|
| Advance | 1438 | 15 | 0.525 | 6.20 | 0.60 | 9.20 |
| Delay | 255 | 9 | 0.055 | 5.84 | -5.40 | 5.10 |
| Same time | 477 | 12 | undefined | 2.52 | -2.40 | 0.00 |

Advance prediction explains much more within-group variance. Delays are systematically underestimated and do not beat predicting zero on MAE. Same-time cases receive a modest negative prediction on average; their R² is undefined because the observed target has zero variance.

Unweighted sign counts: 1,335/1,438 advances and 145/255 delays have the correct predicted sign. Equal-network-weighted sign rates differ (96.3% and 37.9%) because category counts vary strongly across networks. No exact-zero requirement should be used as an accuracy measure for a continuous prediction.

Full pair-weighted and equal-network-weighted tables, including without-54 results, are saved in `timing_direction_scores.csv`.